Imports & Relative Path Setup

In [ ]:
import os
import shutil
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image

# --- CONFIGURATION ---
# Raw Download Folder (Input)
SOURCE_ROOT = Path("Kvasir-SEG") 

# Cleaned Dataset Folder (Output)
DATA_ROOT = Path("data")

# Split Ratios
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 42

print(f"🔧 Project Root: {Path.cwd()}")

Verify Data

In [ ]:
def verify_raw_data(source_dir):
    """Checks if raw Kvasir-SEG data is valid before we split it."""
    image_dir = source_dir / "images"
    mask_dir = source_dir / "masks"
    
    if not image_dir.exists() or not mask_dir.exists():
        print(f"⚠️ Raw data not found at {source_dir}. Skipping verification.")
        return False

    images = sorted([f.name for f in image_dir.iterdir() if f.name != ".DS_Store"])
    masks = sorted([f.name for f in mask_dir.iterdir() if f.name != ".DS_Store"])
    
    # 1. Count Check
    if len(images) != len(masks):
        print(f"❌ Mismatch: {len(images)} images vs {len(masks)} masks")
        return False
        
    # 2. Spot Check (First 5)
    print(f"🔍 Verifying alignment on first 5 samples...")
    for img_name in images[:5]:
        img_p = str(image_dir / img_name)
        mask_p = str(mask_dir / img_name)
        
        if cv2.imread(img_p).shape[:2] != cv2.imread(mask_p).shape[:2]:
            print(f"❌ Dimension mismatch: {img_name}")
            return False

    print("✅ Raw data looks good!")
    return True

# Run Verification
verify_raw_data(SOURCE_ROOT)

Split Data

In [ ]:
def create_split():
    # 1. Safety Check: Don't overwrite if it already exists
    if DATA_ROOT.exists():
        print(f"✅ Data split already exists at '{DATA_ROOT}'. Skipping split generation.")
        return

    print(f"🚀 Generating splits from {SOURCE_ROOT}...")
    
    if not SOURCE_ROOT.exists():
        raise FileNotFoundError(f"❌ Error: Raw folder '{SOURCE_ROOT}' not found. Did you unzip the file?")

    # 2. Reproducibility
    random.seed(SEED)
    
    # 3. Get file list
    src_img_dir = SOURCE_ROOT / "images"
    src_mask_dir = SOURCE_ROOT / "masks"
    all_files = sorted([f.name for f in src_img_dir.iterdir() if f.name != ".DS_Store"])
    random.shuffle(all_files)
    
    # 4. Calculate Indices
    total = len(all_files)
    train_end = int(total * TRAIN_RATIO)
    val_end = train_end + int(total * VAL_RATIO)
    
    splits = {
        "train": all_files[:train_end],
        "val":   all_files[train_end:val_end],
        "test":  all_files[val_end:]
    }
    
    # 5. Copy Files
    for split_name, files in splits.items():
        # Make directories (e.g., data/train/images)
        (DATA_ROOT / split_name / "images").mkdir(parents=True, exist_ok=True)
        (DATA_ROOT / split_name / "masks").mkdir(parents=True, exist_ok=True)
        
        print(f"   - Processing {split_name}: {len(files)} images...")
        for name in files:
            shutil.copy2(src_img_dir / name, DATA_ROOT / split_name / "images" / name)
            shutil.copy2(src_mask_dir / name, DATA_ROOT / split_name / "masks" / name)
            
    print("✅ Dataset split complete.")

# Run Split
create_split()

Data Set Creation

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
%pip install torch torchvision
from torch.utils.data import Dataset, DataLoader # Assuming PyTorch usage
from PIL import Image

# --- RELATIVE PATH SETUP ---
# Get the directory where this notebook is running
BASE_DIR = Path.cwd()

# Define the path to your data folder relative to this notebook
DATA_DIR = BASE_DIR / "data"

print(f"✅ Project Root: {BASE_DIR}")
print(f"📂 Data Directory: {DATA_DIR}")

# Check if it exists
if not DATA_DIR.exists():
    print("❌ Error: 'data' folder not found. Did you run the split script?")
else:
    print("Found 'data' folder. Ready to load.")

Dataset Class

In [ ]:
class KvasirDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.img_dir = self.root_dir / split / "images"
        self.mask_dir = self.root_dir / split / "masks"
        
        if not self.img_dir.exists():
            raise FileNotFoundError(f"Split '{split}' not found in {root_dir}")
        
        self.images = sorted([f.name for f in self.img_dir.iterdir() if f.name != ".DS_Store"])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = str(self.img_dir / img_name)
        mask_path = str(self.mask_dir / img_name)
        
        # Load Image & Mask
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Binarize
        _, mask = cv2.threshold(mask, 127, 1.0, cv2.THRESH_BINARY)
        mask = mask.astype(np.float32)

        return image, mask

Visualization

In [ ]:
try:
    # Use the global DATA_ROOT variable
    train_dataset = KvasirDataset(root_dir=DATA_ROOT, split="train")
    
    # Grab a random sample
    idx = random.randint(0, len(train_dataset)-1)
    img, mask = train_dataset[idx]
    
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Image")
    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Mask")
    plt.show()
    
except Exception as e:
    print(f"⚠️ Could not load data: {e}")